In [6]:
import pydantic
pydantic.__version__

'2.12.5'

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

ahmed = User(name='ahmed', age='25') #This work in pydantic

mohamed = User(name='ahmed', age=None)

print(mohamed)

ValidationError: 1 validation error for User
age
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type

In [ ]:
from textwrap import indent
from pydantic import ValidationError

class User(BaseModel):
    uid: int
    username: str
    email: str
    bio: str = ""
    is_active: bool = True
    full_name: str | None = None
    verified_at: datetime | None = None



#user = User()
# will raise an error 
user = User(uid=1, username='momo', email="momo@gmai.com")
print(user.model_dump_json(indent=2))

{
  "uid": 1,
  "username": "momo",
  "email": "momo@gmai.com",
  "bio": "",
  "is_active": true,
  "full_name": null,
  "verified_at": null
}


In [28]:
try:
    user = User(username=None, email="momo@gmai.com")
except ValidationError as e:
    print(f" your error: \n {e}")


 your error: 
 2 validation errors for User
uid
  Field required [type=missing, input_value={'username': None, 'email': 'momo@gmai.com'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
username
  Input should be a valid string [type=string_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


In [ ]:
from ast import alias
import email
from email.policy import default, strict
import random
from turtle import mode
from typing import Literal
from pydantic import Field, EmailStr, SecretStr, field_validator, model_validator,computed_field, ConfigDict
from datetime import datetime, UTC
from functools import partial
from typing import Annotated
from uuid import UUID, uuid4


class BlogPost(BaseModel):
    model_config = ConfigDict(populate_by_name=True, strict=True) # if an int is '123' now it will raise an error
    ubid: UUID = Field(alias='id',default_factory=uuid4)
    email: EmailStr
    title: str
    author: User
    password: SecretStr
    c_password: SecretStr
    content: str
    views: int = 0
    tags: list[str] = Field(default_factory=list)
    created: datetime = Field(default_factory=lambda: datetime.now(tz=UTC)) # to run this each time a blog is created not when a class is created
    created2: datetime = Field(default_factory=partial(datetime.now,tz=UTC)) #same thing
    author_id: str | Annotated[int, Field(gt=1)]
    status: Literal["pub", "arch"] = 'pub'

    @field_validator('password', mode="before") #checking before processing
    @classmethod
    def pass_check(cls, v):
        if not v:
            v = ''.join(str(random.randint(1,9)) for _ in range(8))
            return v
        return v
    @model_validator(mode="after") # get the whole object so u can do what u want
    def check(self):
        if self.password != self.c_password:
            raise ValueError("password don't match")
        return self
    @computed_field
    @property
    def viral(self) -> bool: # tyoe hint here is essential other it will error
        return self.views >= 100

tagss=('m','n')
try:
    blog = BlogPost(title="tit",
    id='2e369c12-a513-4e09-8f59-68096dd2edcd',
    content='cont',
    author_id=2,
    status="pub",
    email="mom@gmai.com",
    password="11",
    c_password="11",
    author=user,
    tags=[*tagss])
except ValidationError as e:
    print(e)


print(blog.password.get_secret_value())
print(blog.model_dump_json(indent=2, by_alias=True, exclude={'title'}))

11
{
  "id": "2e369c12-a513-4e09-8f59-68096dd2edcd",
  "email": "mom@gmai.com",
  "author": {
    "uid": 2,
    "username": "momo",
    "email": "momo@gmai.com",
    "bio": "",
    "is_active": true,
    "full_name": null,
    "verified_at": null
  },
  "password": "**********",
  "c_password": "**********",
  "content": "cont",
  "views": 0,
  "tags": [
    "m",
    "n"
  ],
  "created": "2026-05-11T12:44:54.247473Z",
  "created2": "2026-05-11T12:44:54.247473Z",
  "author_id": 2,
  "status": "pub",
  "viral": false
}
